# Notebook 02 — Nettoyage des 3 sources de données (affichage des résultats)

> ⚠️ **Ce notebook ne calcule plus rien.** Depuis la réorganisation du projet, tout le
> calcul est fait par le script `scripts/etape02_nettoyage_donnees.py`, à lancer **à la main**, depuis la racine du
> projet :
>
> ```bash
> python scripts/etape02_nettoyage_donnees.py
> ```
>
> Ce notebook se contente de **lire et afficher** ce que ce script a produit : les gros
> fichiers de sortie (chemins dans `config.py`) et le rapport d'exécution `02_nettoyage`
> (`outputs/rapports/`, voir `rapports.py`), qui contient tous les compteurs et petits
> tableaux de diagnostic autrefois imprimés au fil des cellules.
>
> Il est donc **léger et ré-exécutable à volonté** (`Run All` en quelques secondes), sans
> jamais relancer un calcul. Si tu changes un paramètre dans `config.py`, relance d'abord
> le script, puis ce notebook.


**Ce que le script a fait**, en 3 parties indépendantes (aucune ne lit le résultat d'une
autre) :

- **Partie A — Caractéristiques** (`datashare.parquet`) : période de départ, filtre
  automatique des caractéristiques trop incomplètes, nettoyage du secteur (`sic2`)
  → `data/interim/characteristics_clean.parquet` + `caracteristiques_retenues.json`
- **Partie B — Rendements** (`StockReturn.parquet`) : conversion des codes CRSP non
  numériques, suppression des rendements manquants (jamais imputés, c'est la cible)
  → `data/interim/returns_clean.parquet`
- **Partie C — Macro** (`MacroData.parquet`) : restriction à la période utile,
  construction des 8 prédicteurs macro de GKX
  → `data/interim/macro_clean.parquet`

## 0. Import et chargement du rapport d'exécution

In [ ]:
import sys
sys.path.append("..")  # config.py, rapports.py sont a la racine du projet

import pandas as pd
import matplotlib.pyplot as plt

import config
import rapports

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

# Leve une erreur explicite (avec la commande a lancer) si le script n'a jamais tourne.
rap = rapports.charger('02_nettoyage')
print(rap.resume())

## Partie A — Caractéristiques (`datashare.parquet`)

### A.1 / A.2 Point de départ et choix de la période

Beaucoup de caractéristiques comptables ne sont fiables qu'à partir des années 1970-1980
(couverture Compustat plus complète) : démarrer en 1957 comme GKX ajouterait surtout des
lignes très incomplètes. Le tableau ci-dessous montre le taux moyen de valeurs manquantes
par décennie, qui motive le choix de `ANNEE_DEBUT`.

👉 **Pour ajuster la valeur**, modifie `ANNEE_DEBUT` dans `config.py`, puis relance le
script (la partie C utilisera automatiquement la même valeur).

In [ ]:
print("Dimensions de depart :", tuple(rap.valeur('A_shape_depart')))
print()
print("Taux moyen de valeurs manquantes par decennie (%) :")
display(rap.table('A_missing_par_decennie'))

print(f"ANNEE_DEBUT (config.py) : {rap.valeur('A_annee_debut')}")
print("Dimensions apres filtre sur l'annee :", tuple(rap.valeur('A_shape_apres_filtre_annee')))
rap.table('A_apercu_brut')

### A.3 / A.3bis L'univers candidat et le filtre de valeurs manquantes

`config.CARACTERISTIQUES` liste les 94 noms de colonnes de `datashare.parquet` (l'univers
**candidat**). Le script en retient automatiquement un **sous-ensemble** : il calcule le
taux de valeurs manquantes de chacune sur `annee >= ANNEE_DEBUT` et exclut celles au-dessus
de `config.SEUIL_MAX_PCT_MANQUANT_CARACTERISTIQUES`.

La liste retenue est sauvegardée dans `data/interim/caracteristiques_retenues.json` —
c'est ce fichier que lisent ensuite automatiquement l'étape 03 et les modèles 04 à 06
(via `config.CARACTERISTIQUES_RETENUES`), sans rien recopier à la main.

In [ ]:
candidates = rap.valeur('A_caracteristiques_candidates')
gardees = rap.valeur('A_caracteristiques_gardees')
exclues = rap.valeur('A_caracteristiques_exclues')
seuil = rap.valeur('A_seuil_missing')

print(f"Univers candidat : {len(candidates)} caracteristiques")
introuvables = rap.valeur('A_colonnes_introuvables')
print("Colonnes introuvables dans le fichier :", introuvables if introuvables else "aucune (OK)")
print()
print(f"Seuil d'exclusion : {seuil*100:.0f}% de valeurs manquantes")
print(f"Caracteristiques RETENUES : {len(gardees)} / {len(candidates)}")
print()
if exclues:
    taux = rap.table('A_taux_missing')['pct_manquant']
    print("Caracteristiques EXCLUES (trop incompletes) :")
    for c in exclues:
        print(f"  - {c:<18s} {taux[c]:5.1f}% manquant")
else:
    print("Aucune caracteristique exclue : toutes les candidates passent le seuil.")

In [ ]:
# Les 25 candidates les plus incompletes, avec le seuil d'exclusion en repere
taux = rap.table('A_taux_missing')['pct_manquant']
top_missing = taux.sort_values(ascending=False).head(25).sort_values()
couleurs = ['tab:red' if v > seuil * 100 else 'tab:blue' for v in top_missing]

fig, ax = plt.subplots(figsize=(8, max(4, 0.28 * len(top_missing))))
ax.barh(top_missing.index, top_missing.values, color=couleurs)
ax.axvline(seuil * 100, color='black', linewidth=1, linestyle='--',
           label=f"Seuil ({seuil*100:.0f}%)")
ax.set_xlabel("% de valeurs manquantes (annee >= ANNEE_DEBUT)")
ax.set_title("25 caracteristiques candidates les plus incompletes\n(rouge = exclue, bleu = retenue)")
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

### A.4 / A.5 Valeurs manquantes restantes et secteur

Les valeurs manquantes résiduelles ne sont **pas** imputées ici : c'est volontaire, elles
le seront à l'étape 03 (partie B), une fois l'univers investissable filtré, pour que la
médiane utilisée vienne de la population réellement modélisée.

`sic2` est transformé en catégorie texte, avec une catégorie explicite `"Inconnu"` plutôt
que d'exclure les entreprises sans secteur renseigné.

In [ ]:
print("Dimensions apres reduction des colonnes :", tuple(rap.valeur('A_shape_apres_reduction')))
print()
print("Valeurs manquantes restantes (% des lignes, imputees a l'etape 03) :")
manquants = rap.table('A_missing_restant_pct')
display(manquants if len(manquants) else "Aucune valeur manquante restante.")

print("Repartition des secteurs (top 10) :")
display(rap.table('A_repartition_secteurs'))

### A.6 Résultat final de la partie A

In [ ]:
print("Dimensions finales :", tuple(rap.valeur('A_shape_finale')))
print("Colonnes finales   :", rap.valeur('A_colonnes_finales'))
print()
print("Valeurs manquantes restantes par colonne :")
display(rap.table('A_missing_final'))
print("Fichier ecrit par le script :", config.FICHIER_CARACTERISTIQUES_CLEAN)

In [ ]:
rap.table('A_describe')

In [ ]:
rap.table('A_apercu_final')

## Partie B — Rendements (`StockReturn.parquet`)

### B.1 à B.3 Codes CRSP non numériques

La colonne `RET` de CRSP mélange des rendements numériques et des **codes texte spéciaux**
indiquant une raison précise pour laquelle le rendement est manquant (pas de cotation,
donnée insuffisante, société retirée de la cote...). Dans tous les cas, ces lignes n'ont
pas de rendement utilisable : le script les convertit en `NaN`.

In [ ]:
print("Dimensions de depart :", tuple(rap.valeur('B_shape_depart')))
print("Type de la colonne RET :", rap.valeur('B_dtype_ret_depart'))
print()
print("Codes non numeriques trouves dans RET :")
display(rap.table('B_codes_speciaux'))
print(f"{rap.valeur('B_nb_codes_speciaux')} lignes ({rap.valeur('B_pct_codes_speciaux'):.2f} %)")
print()
print(f"% de RET manquant apres conversion : {rap.valeur('B_pct_manquant_apres_conversion'):.2f} %")

### B.4 Rendements manquants : supprimés, jamais imputés

**Règle importante, différente de la partie A :** on n'impute jamais la variable cible.
Remplacer un rendement manquant par une médiane reviendrait à *inventer* la donnée qu'on
cherche justement à prédire — une fuite de données qui fausserait complètement
l'évaluation du modèle. La seule option raisonnable est de **retirer** ces lignes.

In [ ]:
print(f"Lignes avant suppression : {rap.valeur('B_lignes_avant_dropna')}")
print(f"Lignes apres suppression : {rap.valeur('B_lignes_apres_dropna')}")
print(f"Lignes supprimees : {rap.valeur('B_pct_lignes_supprimees'):.2f} %")

### B.5 Valeurs extrêmes (diagnostic uniquement)

Un rendement mensuel de +500 % ou -95 % est rare mais **pas forcément une erreur** (delisting,
fusion, action très volatile). Le script se contente donc de les repérer, sans rien supprimer.

In [ ]:
print("Statistiques descriptives de RET :")
display(rap.table('B_describe_ret'))
print("Quantiles extremes :")
display(rap.table('B_quantiles_ret'))

seuil_ex = rap.valeur('B_seuil_extreme')
print(f"Rendements > +{seuil_ex*100:.0f}% : {rap.valeur('B_nb_extremes_hauts')} lignes")
print(f"Rendements < -95%   : {rap.valeur('B_nb_extremes_bas')} lignes")

### B.6 / B.7 Harmonisation et résultat final de la partie B

In [ ]:
doublons = rap.valeur('B_doublons_permno_mois')
print(f"Doublons (permno, annee_mois) : {doublons}")
if doublons > 0:
    print("ATTENTION : a investiguer avant la fusion (etape 03, partie A).")
print()
print("Dimensions finales :", tuple(rap.valeur('B_shape_finale')))
print("Colonnes finales   :", rap.valeur('B_colonnes_finales'))
print("Periode couverte   : de", rap.valeur('B_periode')[0], "a", rap.valeur('B_periode')[1])
display(rap.table('B_missing_final'))
rap.table('B_apercu_final')

## Partie C — Variables macro (`MacroData.parquet`)

### C.1 à C.3 Conversion numérique et comblement des trous

Les fichiers Welch & Goyal contiennent souvent de grands nombres formatés avec une virgule
comme séparateur de milliers (ex: `"4,796.56"`), que pandas ne convertit pas automatiquement :
le script force la conversion après avoir retiré les virgules.

Contrairement aux caractéristiques d'entreprises, il n'y a qu'**une seule ligne par mois**
ici : pas de médiane croisée possible. Le comblement se fait donc par **dernière valeur
connue** (`ffill`, standard pour des séries macro qui évoluent lentement), avec un `bfill`
en secours pour un éventuel trou en tout début de période.

In [ ]:
print("Dimensions de depart :", tuple(rap.valeur('C_shape_depart')))
print("Colonnes :", rap.valeur('C_colonnes_depart'))
print("Periode couverte (yyyymm) :", rap.valeur('C_periode_depart'))
print()
print("Types apres conversion numerique forcee :")
display(rap.table('C_dtypes_apres_conversion'))
print("Dimensions apres filtre sur l'annee :", tuple(rap.valeur('C_shape_apres_filtre_annee')))

In [ ]:
print("% de valeurs manquantes par colonne AVANT comblement :")
display(rap.table('C_missing_avant_comblement'))

vides = rap.valeur('C_colonnes_entierement_vides')
if vides:
    print("ATTENTION - colonnes ENTIEREMENT vides sur la periode choisie :", vides)
    print("-> soit tu avances ANNEE_DEBUT, soit tu ne pourras pas utiliser ces colonnes.")
else:
    print("Comblement OK : aucune colonne entierement vide.")

### C.4 Les 8 prédicteurs macro de Gu, Kelly & Xiu (2020)

| Prédicteur | Colonne finale | Calcul |
|---|---|---|
| `dp` | `macro_dp` | `log(D12) - log(Index)` |
| `ep` | `macro_ep` | `log(E12) - log(Index)` |
| `bm` | `macro_bm` | colonne `b/m` |
| `ntis` | `macro_ntis` | colonne `ntis` |
| `tbl` | `macro_tbl` | colonne `tbl` |
| `tms` | `macro_tms` | `lty - tbl` |
| `dfy` | `macro_dfy` | `BAA - AAA` |
| `svar` | `macro_svar` | colonne `svar` |

⚠️ **Pourquoi le préfixe `macro_` ?** Les caractéristiques d'entreprise de la partie A
contiennent déjà des colonnes `bm` et `ep`. Sans préfixe, la fusion de l'étape 03
renommerait silencieusement les deux versions en `bm_x` / `bm_y` — un piège classique.

In [ ]:
print("Valeurs <= 0 dans les colonnes passees au log (le log y serait NaN) :")
print(rap.valeur('C_valeurs_non_positives'))
print()
display(rap.table('C_apercu_predicteurs'))
print("Valeurs manquantes dans les 8 predicteurs GKX :")
display(rap.table('C_missing_predicteurs_gkx'))

### C.5 / C.6 Résultat final de la partie C

`Rfree` est conservé séparément (ce n'est pas un prédicteur, mais l'étape 03 en a besoin
pour calculer le rendement excédentaire `excess_return = RET - Rfree`).

In [ ]:
print("Dimensions finales :", tuple(rap.valeur('C_shape_finale')))
print("Colonnes finales   :", rap.valeur('C_colonnes_finales'))
print()
print("Valeurs manquantes restantes :")
display(rap.table('C_missing_final'))
rap.table('C_apercu_final')

## Résumé global

Les 3 fichiers nettoyés sont dans `data/interim/` :

- `characteristics_clean.parquet` (Partie A) — peut encore contenir des valeurs manquantes
  (volontaire, imputées à l'étape 03)
- `returns_clean.parquet` (Partie B) — plus aucune valeur manquante dans `RET`
  (lignes supprimées, jamais imputées)
- `macro_clean.parquet` (Partie C) — plus aucune valeur manquante (`ffill`/`bfill`)

**Étape suivante :** `python scripts/etape03_construction_panel.py`, puis
`03_construction_panel.ipynb` pour en visualiser le résultat.